#### BRONZE LAYER

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/tosinforlly@gmail.com/fmcg_project/1_setup/utilities

In [0]:
dbutils.widgets.text('catalog', 'spotizone', 'Catalog')
dbutils.widgets.text('data_source', 'products', 'Data Source')


In [0]:
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

storage_path = f's3://sport-bar/{data_source}/*.csv'

In [0]:
df_prod = (
    spark.read.format("csv")
    .option("inferSchema", True)
    .option("header", True)
    .load(storage_path)
    .withColumn("ingest_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name")
)

In [0]:
df_prod.write\
    .mode("overwrite")\
    .format("delta")\
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')

#### SILVER LAYER

In [0]:
df = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source}')
display(df)

In [0]:
df.printSchema()

- 1. Drop Duplicates

In [0]:
# Check Number of Rows Before Duplicate
du_check = df.groupBy(['product_id']).count().filter(F.col('count') > 1)

# Remove Duplicates in Product_id
df_silver = df.dropDuplicates(['product_id'])

In [0]:
# Quality Check
print(f'Number of Rows Before Removing Duplicates: {df.count()}')
print(f'Number of Rows After Removing Duplicates: {df_silver.count()}')


In [0]:
display(df_silver)

- 2. Remove Whitespaces in all String Attributes

In [0]:
df_silver = (
    df_silver
    .withColumn('product_name', F.trim(F.col('product_name')))
    .withColumn('product_id', F.trim(F.col('product_id')))
    .withColumn('category', F.trim(F.col('category')))
)

display(df_silver)

- 3. Case Fixing

In [0]:
# Standardizing case in category just like in parent company
df_silver = df_silver.withColumn(
    'category',
    F.when(F.col('category').isNull(), None)
    .otherwise(F.initcap(F.col('category'))))

df_silver.select('category').display()

- 4. Spelling Fixing

In [0]:
# Correct protein Spelling in Product name and category
df_silver = (
    df_silver
    .withColumn('product_name', F.regexp_replace(F.col('product_name'), '(?i)Protien', 'Protein'))
    .withColumn('category', F.regexp_replace(F.col('category'), '(?i)Protien', 'Protein'))
)

display(df_silver)

- 5. 'Division' Column Creation

In [0]:
# Category - Division Mapping From Business Team

div_map = {
    'Energy Bars': 'Nutrition Bars',
    'Protein Bars': 'Nutrition Bars',
    'Granola & Cereals': 'Breakfast Foods',
    'Recovery Dairy': 'Diary & Recovery',
    'Healthy Snacks': 'Healthy Snacks',
    'Electrolyte Mix': 'Hydration & Electrolyte'
}

div_map_df = spark.createDataFrame(
    [(k,v) for k,v in div_map.items()],
    ['category', 'division']
)

display(div_map_df)

In [0]:
df_silver = df_silver.join(div_map_df, "category", "left")

In [0]:
display(df_silver)

- 6. 'Variant' Column Creation

In [0]:
df_silver = df_silver.withColumn('variant', F.regexp_extract(F.col('product_name'), r'\((.*?)\)', 1))

In [0]:
display(df_silver)

- 7. 'Product_id' Inconsistency

In [0]:
# Invalid product_ids are replaced with a fallback value to avoid losing fact records and ensure downstream joins remain valid

df_silver = (
    df_silver
    # 1. Generate deterministic product_code from product_name
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    # 2. Clean product_id: keep only numeric IDs, else set to 999999
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
    # 3. Rename product_name → product
    .withColumnRenamed("product_name", "product")
)

In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.select('product_code','division','category','product', 'variant' ,'product_id','ingest_timestamp','file_name')

display(df_silver)
# DBTITLE 1 )

%md
##### Write

In [0]:
df_silver.write\
    .format('delta')\
    .mode('overwrite')\
    .option('enableChangeDataFeed', 'true')\
    .option('mergeSchema', 'true')\
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')

#### GOLD LAYER

In [0]:
# Call From Silver Layer
df_silver = spark.sql(f'SELECT * FROM {catalog}.{silver_schema}.{data_source}')

In [0]:
df_gold = df_silver.select('product_code', 'division', 'category', 'product', 'variant')
display(df_gold)

In [0]:
df_gold.write\
    .format('delta')\
    .mode('overwrite')\
    .option('enableChangeDataFeed', 'true')\
    .saveAsTable(f'{catalog}.{gold_schema}.sb_dim_{data_source}')

##### Merge With Parent Company

In [0]:
%sql
CONVERT TO DELTA spotizone.gold.dim_products

In [0]:
# Assigning Source and Target Variables
parent_products = DeltaTable.forName(spark, 'spotizone.gold.dim_products')
child_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM spotizone.gold.sb_dim_products;")

display(child_products.limit(5))

In [0]:
(
    parent_products.alias("target")
    .merge(
        source=child_products.alias("source"),
        condition="target.product_code = source.product_code"
    )
    .whenMatchedUpdate(
        set={
            "division": "source.division",
            "category": "source.category",
            "product": "source.product",
            "variant": "source.variant"
        }
    )
    .whenNotMatchedInsert(
        values={
            "product_code": "source.product_code",
            "division": "source.division",
            "category": "source.category",
            "product": "source.product",
            "variant": "source.variant"
        }
    )
    .execute()
)
